This notebook demonstrates how to do custom object detection using the UGOT camera.

**Prerequisites**: 
- Python 3.
- A Google account is recommended to access Google Colab's GPU for training.
- A Roboflow account is recommended for annotation, though you may use other annotation tools as you wish.

**Note on Google Colab**: The UGOT cannot be connected to in Google Colab, so you will have to collect the images and test the video feed using a local Python kernel on your computer. However, to train the model, it is highly recommended to use Google Colab's (free) resources to speed up training time.

Unfortunately, as the Google Colab extension in VS Code currently does not support access to Google Drive, and also cannot access local files, you will need to use Google Colab in the browser.

In [ ]:
%pip install ultralytics ugot opencv-python

In [16]:
import cv2
import numpy as np
import time

from IPython.display import clear_output

from ultralytics import YOLO

from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.120") # replace this with your UGOT IP address
got.open_camera()

192.168.1.120:50051


The following few cells allow you to see how the pretrained YOLO model sees objects before you train the model on your own data. This must be done locally, as Google Colab cannot connect to the UGOT camera.

In [7]:
# Load pretrained model from YOLO 
model = YOLO("yolo11n.pt")

In [17]:
# Helper: Draw bounding boxes
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes  # bounding boxes

        for box in boxes:
            # xyxy format: [x1, y1, x2, y2]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            # Confidence & label
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = r.names[cls_id]

            # Draw rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2)
    return frame

In [15]:
# Visualize bounding boxes with live video feed
while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Run YOLO detection
        results = model(img, verbose=False)

        # Draw output
        output = draw_detections(img, results)

        # Show
        cv2.imshow("YOLO Detection", output)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cv2.destroyAllWindows()


To train the YOLO model using our own images, we need to first capture some images of the object we want to detect.
The following cell will allow the UGOT to auto capture an image every half second.
Here are some tips for good training data:
- Capture at least 50 images. The more, the better, but remember you will also need to annotate them.
- Move the UGOT / objects around so that you get images when the object is near/far, under different lighting conditions, partially obscured, multiple objects in the same picture, etc. Think about what kind of conditions the robot might encounter during the competition.

In [17]:
# Save images from UGOT video feed at regular time intervals
from ugot import ugot
import cv2
import numpy as np
import time
import os

SAVE_DIR = "captured_demo" # changed folder for demo purposes
os.makedirs(SAVE_DIR, exist_ok=True)

got = ugot.UGOT()
got.initialize("192.168.1.231")
got.open_camera()

counter = 17    # with multiple runs, change this to one after the last captured image name to avoid overwriting images
interval = 1   # seconds between captures

print("Auto-capturing images. Press 'q' to stop.")

last_time = time.time()

try:
    while True:
        frame = got.read_camera_data()
        if frame is not None:
            nparr = np.frombuffer(frame, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            cv2.imshow("UGOT Camera", img)

            # Auto-save
            if time.time() - last_time >= interval:
                filename = f"{SAVE_DIR}/img_{counter:04d}.jpg"
                cv2.imwrite(filename, img)
                print(f"Saved: {filename}")
                counter += 1
                last_time = time.time()

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

finally:
    cv2.destroyAllWindows()
    print("Done.")


192.168.1.231:50051
Auto-capturing images. Press 'q' to stop.
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
No camera data received
Saved: captured_demo/img_0017.jpg
Saved: captured_demo/img_0018.jpg
Saved: captured_demo/img_0019.jpg
Saved: captured_demo/img_0020.jpg
Saved: cap

After capturing the images, we need to annotate them with the bounding boxes and class labels. Create an account at [app.roboflow.com](https://app.roboflow.com/).
1. Create a new project, select `Traditional` (not Rapid) tool and `Object Detection` project type, and upload your images.
2. Use the tools to annotate your images (label each object with your classes). If you have created a new account, you should be able to use the built-in AI annotation tool to speed up the process as part of your free trial. Ensure that the bounding boxes are tight.
3. You may apply augmentations in the Dataset tab if you wish. However, augmented images can be generated on the fly by instead modifying the `augments` parameter in `model.train` below.
4. Select a train/test/val split when creating a version of your dataset to download. You must have some images in each. The default 70/20/10 split should do.
5. Under the Versions tab, download the dataset in the YOLOv11 format (select "Download zip to computer", download, and unzip).
6. Upload the folder to your Google Drive.

Now, we need to train the model. Note that since this overwrites the previous model classes, only your new classes that you have annotated will be detected, although we use YOLOv11 as the base model.

1. Open this notebook in Google Colab. In the top right corner of the browser webpage, select "Change runtime type" and select an available GPU (usually T4). If you select CPU, or simply try to run this cell on your computer, the training will take much longer.
2. Change the path in `model.train` below to the correct path to `data.yaml` in your Drive folder. Change the paths of train / val / test in `data.yaml` as well. You can copy the path from the file explorer on the left.

After training, **remember to download the model** from `runs/detect/trainx/weights` where `x` is the $x$th time you have trained a model in the current runtime. The weights will be deleted once the runtime disconnects. You probably only need `best.pt`.

In [ ]:
# Allow Google Colab to access your Google Drive files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use pretrained model from YOLO
model = YOLO("yolo11n.pt")

# Train on new roboflow data
model.train(data="/content/drive/MyDrive/roboflow_coffee/data.yaml", epochs=50, imgsz=512)
# model.train(data="./roboflow_bad/data.yaml", epochs=50, imgsz=512)

Now test your model with the new objects (remember to switch back to local and run imports and definitions):

In [ ]:
trained = YOLO("best_token.pt")

while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Run YOLO detection
        results = trained(img, verbose=False)

        # Draw output
        output = draw_detections(img, results)

        # Show
        cv2.imshow("YOLO Detection - With Custom Object", output)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cv2.destroyAllWindows()

Now you can use the model!

Here is a sample program using the UGOT engineer robot:
- The UGOT will follow a line continuously.
- At an intersection, the UGOT will stop and check for an object.
- If the UGOT sees a certain object, it will follow the directions, e.g. choose the left path if it sees a candle; pick up a token and turn around.

In [9]:
def line_follow_camera():
    got.open_camera()

    # got.load_models(["line_recognition"])
    got.set_track_recognition_line(line_type = 0)

    line_info = got.get_single_track_total_info()  # list: [offset, type, x, y]
    offset = line_info[0]
    line_type = line_info[1]

    try:
        while True:
            frame = got.read_camera_data()
            if frame is not None:
                nparr = np.frombuffer(frame, np.uint8)
                img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

                # Run YOLO detection
                results = model(img, verbose=False)

                # Draw output
                output = draw_detections(img, results)

                # Show
                cv2.imshow("YOLO Detection - With Custom Object", output)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            line_info = got.get_single_track_total_info()
            offset = line_info[0]
            line_type = line_info[1]

            if line_type != 1: # no line, or intersection, or crossroads
                return line_type, results

            degrees = int(offset / 4)
            got.mecanum_move_xyz(0, 20, degrees)
            time.sleep(0.1)

    finally:
        got.mecanum_stop()
        

In [ ]:
# Helper functions: pick up token
def ap_approach():
    # got.load_models(["apriltag_qrcode"])
    while True:
        AP_info = got.get_apriltag_total_info()
        if AP_info:
            x_coord = AP_info[0][1]
            dist = AP_info[0][6]
            if x_coord < 290:
                 got.mecanum_move_xyz(-3, 3, 0)
            elif x_coord > 350:
                got.mecanum_move_xyz(3, 3, 0)
            elif dist > 0.12:
                got.mecanum_move_xyz(0, 3, 0)
            else:
                got.mecanum_stop()
                break
        else:
            got.mecanum_stop()
    print("Stopped.")

def pickup_ap():
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(0, 0, -70, 800) #down - for apriltag
    time.sleep(1)
    got.mechanical_clamp_close()
    time.sleep(2)
    got.mechanical_joint_control(0, 30, -50, 800) #up

def put_ap():
    got.mechanical_joint_control(-90, 30, -50, 800)
    time.sleep(1)
    got.mechanical_joint_control(-90, -20, -30, 800)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(2)

In [4]:
# MAIN CODE
model = YOLO("best.pt")

got.load_models(["line_recognition", "apriltag_qrcode"])

num_intersections = 0

try:
    while num_intersections < 2:
        line_type, results = line_follow_camera()
        if line_type == 2:
            num_intersections += 1
            for r in results:
                # ['candle', 'coffee', 'coke', 'token']

                print(r.boxes)
                detected = r.boxes.cls.tolist()
                if 0 in detected: # candle
                    got.mecanum_turn_speed_times(2, 40, 20, 2)
                    got.mecanum_translate_speed_times(0, 10, 20, 1)
                elif 3 in detected: # token
                    ap_approach()
                    pickup_ap()
                    got.mecanum_turn_speed_times(2, 40, 180, 2)
                else:
                    got.mecanum_turn_speed_times(3, 40, 20, 2)
finally:
    got.mecanum_stop()
    cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'

We can also use properties of the bounding boxes surrounding the objects to control robot behaviour.
For example, the code in the following cell will get the self-balancing car to look for a specific object, and then go to it.

If you have only trained your model to recognise a single object, the class will probably be `0`.
Otherwise, the index can be found from the `names` list in the `data.yaml` file that roboflow generated, or you can view the `cls` attribute in `r.boxes` printed in the cell above.

More information about the properties of the results/boxes can be found in the [ultralytics documentation](https://docs.ultralytics.com/reference/engine/results/#ultralytics.engine.results.Boxes).

In [20]:
# Centralise and approach an object to a certain distance 
import cv2
import numpy as np
import time

from ultralytics import YOLO

from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.231")

got.open_camera()

trained = YOLO("best_coffee.pt")

# Helper: Draw bounding boxes
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes  # bounding boxes

        for box in boxes:
            # xyxy format: [x1, y1, x2, y2]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            # Confidence & label
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = r.names[cls_id]

            # Draw rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2)
    return frame

got.balance_start_balancing()
time.sleep(1)

def find_object():
    try:
        while True:
            frame = got.read_camera_data()
            if frame is not None:
                nparr = np.frombuffer(frame, np.uint8)
                img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

                # Run YOLO detection
                results = trained(img, verbose=False)

                # Draw output
                output = draw_detections(img, results)

                ############
                # reset variables in case candle is out of frame
                area = 1000
                x = 0.5
                max_conf = 0.0
                max_idx = -1
                
                # Find candle with highest confidence
                for r in results:
                    detected = r.boxes.cls.tolist()
                    confidences = r.boxes.conf.tolist()
                    xywhn = r.boxes.xywhn.tolist()
                    
                    for idx, cls_id in enumerate(detected):
                        if cls_id == 0:  # candle
                            conf = confidences[idx]
                            if conf > max_conf:
                                max_conf = conf
                                max_idx = idx
                                x, y, w, h = xywhn[max_idx]
                                area = w * h
                
                # Move toward highest confidence candle
                if max_idx != -1:  # candle found
                    cv2.putText(output, f"Centre: ({x:.2f}, {y:.2f})", (30, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    cv2.putText(output, f"Area: {area:.3f}", (30, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    cv2.putText(output, f"Max confidence: {max_conf:.3f}", (30, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    
                    if x > 0.6:
                        got.balance_move_turn(0, 3, 3, 10)  # turn right while moving forward
                    elif x < 0.4:
                        got.balance_move_turn(0, 3, 2, 10)  # turn left while moving forward
                    else:
                        if area < 0.06:
                            got.balance_move_speed(0, 5)
                        else:
                            got.balance_stop_balancing()
                else:  # candle not in frame
                    got.balance_turn_speed(3, 10)  # turn right to scan for candle
                ############

                # Show
                cv2.imshow("YOLO Detection - With Custom Object", output)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
    finally:
        got.balance_stop_balancing()
        cv2.destroyAllWindows()

192.168.1.231:50051
